In [ ]:
import os
import glob
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from PIL import Image, ImageTk
from io import BytesIO
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2 import preprocess_input as res_pre
import time
import threading
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox, filedialog
from datetime import datetime
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import queue
import gc
import atexit
warnings.filterwarnings('ignore')

# 기본 설정
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# GPU 설정
def setup_gpu():
    try:
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            try:
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
                print(f"[GPU INFO] GPU 사용 가능: {len(gpus)}개")
                print(f"[GPU INFO] 사용 중인 GPU: {gpus[0].name}")
                return True
            except RuntimeError as e:
                print(f"[GPU WARNING] GPU 설정 실패: {e}")
                return False
        else:
            print("[GPU INFO] GPU를 찾을 수 없습니다. CPU를 사용합니다.")
            return False
    except Exception as e:
        print(f"[GPU ERROR] GPU 초기화 오류: {e}")
        return False

GPU_AVAILABLE = setup_gpu()
IMG_SIZE = (224, 224)

class FoodImagePredictor:
    def __init__(self, model_dir='models', timestamp='20250716_144252'):
        self.model_dir = model_dir
        self.timestamp = timestamp
        self.eff_model = None
        self.res_model = None
        self.xgb_model = None
        self.index_to_label = {}
        self.properties_data = {}
        
    def log(self, message, level="INFO"):
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] [{level}] {message}")
    
    def load_models(self):
        try:
            self.log("모델 파일 로드 중...", "INFO")
            if GPU_AVAILABLE:
                self.log("GPU 모드로 모델 로드 중...", "SUCCESS")
                with tf.device('/GPU:0'):
                    self._load_models_on_device()
            else:
                self.log("CPU 모드로 모델 로드 중...", "WARNING")
                with tf.device('/CPU:0'):
                    self._load_models_on_device()
        except Exception as e:
            self.log(f"모델 로드 실패: {e}", "ERROR")
            raise
    
    def _load_models_on_device(self):
        label_map_path = f"{self.model_dir}/label_to_index_{self.timestamp}.joblib"
        if not os.path.exists(label_map_path):
            raise FileNotFoundError(f"라벨 매핑 파일이 없습니다: {label_map_path}")
        
        label_map = joblib.load(label_map_path)
        self.index_to_label = {v: k for k, v in label_map.items()}
        num_classes = len(label_map)
        self.log(f"라벨 매핑 로드 완료 - 클래스 수: {num_classes}", "SUCCESS")
        
        self.log("EfficientNet 모델 로드 중...", "INFO")
        self.eff_model = self.build_model_gpu_optimized(EfficientNetB0, num_classes)
        eff_weights_path = f"{self.model_dir}/effnet_model_best_{self.timestamp}.h5"
        if not os.path.exists(eff_weights_path):
            raise FileNotFoundError(f"EfficientNet 가중치 파일이 없습니다: {eff_weights_path}")
        self.eff_model.load_weights(eff_weights_path)
        self.log("EfficientNet 모델 로드 완료", "SUCCESS")
        
        self.log("ResNet 모델 로드 중...", "INFO")
        self.res_model = self.build_model_gpu_optimized(ResNet50V2, num_classes)
        res_weights_path = f"{self.model_dir}/resnet_model_best_{self.timestamp}.h5"
        if not os.path.exists(res_weights_path):
            raise FileNotFoundError(f"ResNet 가중치 파일이 없습니다: {res_weights_path}")
        self.res_model.load_weights(res_weights_path)
        self.log("ResNet 모델 로드 완료", "SUCCESS")
        
        try:
            self.log("XGBoost 모델 로드 중...", "INFO")
            xgb_path = f"{self.model_dir}/xgb_model_{self.timestamp}.joblib"
            if not os.path.exists(xgb_path):
                raise FileNotFoundError(f"XGBoost 파일이 없습니다: {xgb_path}")
            
            with tf.device('/CPU:0'):
                self.xgb_model = joblib.load(xgb_path)
            
            if hasattr(self.xgb_model, '_Booster'):
                if not hasattr(self.xgb_model, 'use_label_encoder'):
                    self.xgb_model.use_label_encoder = False
                if not hasattr(self.xgb_model, 'eval_metric'):
                    self.xgb_model.eval_metric = 'logloss'
            
            self.log("XGBoost 모델 로드 완료", "SUCCESS")
        except Exception as e:
            self.xgb_model = None
            self.log(f"XGBoost 로드 실패: {e} (CNN만 사용)", "WARNING")
    
    def build_model_gpu_optimized(self, base_cls, num_classes):
        base = base_cls(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
        x = GlobalAveragePooling2D(name='gap')(base.output)
        x = Dense(256, activation='relu')(x)
        x = Dropout(0.2)(x)
        out = Dense(num_classes, activation='softmax', dtype='float32')(x)
        model = Model(inputs=base.input, outputs=out)
        
        if GPU_AVAILABLE:
            try:
                model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
            except:
                pass
        return model
    
    def download_image(self, url):
        try:
            if not url or str(url).strip() == '' or pd.isna(url):
                return None, "URL이 비어있습니다"
            
            url = str(url).strip()
            
            if not url.startswith(('http://', 'https://')):
                return None, f"유효하지 않은 URL 프로토콜: {url[:50]}"
            
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                'Accept': 'image/webp,image/apng,image/*,*/*;q=0.8',
                'Accept-Language': 'ko-KR,ko;q=0.9,en;q=0.8',
                'Accept-Encoding': 'gzip, deflate',
                'Connection': 'keep-alive',
                'Upgrade-Insecure-Requests': '1',
            }
            
            session = requests.Session()
            session.headers.update(headers)
            
            response = session.get(url, timeout=15, verify=False, stream=True)
            response.raise_for_status()
            
            content_type = response.headers.get('content-type', '').lower()
            if content_type and not any(img_type in content_type for img_type in ['image/', 'jpeg', 'jpg', 'png', 'gif', 'webp']):
                return None, f"이미지 파일이 아님: {content_type}"
            
            content_length = response.headers.get('content-length')
            if content_length and int(content_length) > 20 * 1024 * 1024:
                return None, f"파일이 너무 큼: {content_length} bytes"
            
            image_data = BytesIO()
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    image_data.write(chunk)
            
            image_data.seek(0)
            
            try:
                img = Image.open(image_data)
                img.load()
            except Exception as e:
                return None, f"이미지 파싱 오류: {str(e)[:50]}"
            
            if img.width < 30 or img.height < 30:
                return None, f"이미지가 너무 작음: {img.width}x{img.height}"
            
            if img.width > 5000 or img.height > 5000:
                return None, f"이미지가 너무 큼: {img.width}x{img.height}"
            
            if img.mode != 'RGB':
                try:
                    img = img.convert('RGB')
                except Exception as e:
                    return None, f"RGB 변환 실패: {str(e)[:50]}"
            
            return img, "성공"
            
        except requests.exceptions.SSLError as e:
            return None, f"SSL 인증 오류: {str(e)[:50]}"
        except requests.exceptions.Timeout:
            return None, "타임아웃 (15초)"
        except requests.exceptions.ConnectionError as e:
            return None, f"연결 오류: {str(e)[:50]}"
        except requests.exceptions.HTTPError as e:
            return None, f"HTTP 오류: {e.response.status_code}"
        except requests.exceptions.RequestException as e:
            return None, f"네트워크 오류: {str(e)[:50]}"
        except Exception as e:
            return None, f"예상치 못한 오류: {str(e)[:50]}"
    
    def predict_image(self, pil_image):
        try:
            device = '/GPU:0' if GPU_AVAILABLE else '/CPU:0'
            
            with tf.device(device):
                img_array = np.array(pil_image)
                img = tf.image.resize(img_array, IMG_SIZE)
                img = tf.expand_dims(img, axis=0)
                
                inp_eff = eff_pre(img)
                inp_res = res_pre(img)
                p_eff = self.eff_model.predict(inp_eff, verbose=0)
                p_res = self.res_model.predict(inp_res, verbose=0)
                p_cnn = (p_eff + p_res) / 2.0
                
                if self.xgb_model:
                    try:
                        with tf.device('/CPU:0'):
                            feat_eff = Model(self.eff_model.input, self.eff_model.get_layer('gap').output).predict(inp_eff, verbose=0)
                            feat_res = Model(self.res_model.input, self.res_model.get_layer('gap').output).predict(inp_res, verbose=0)
                            feat = np.hstack([feat_eff, feat_res])
                            p_xgb = self.xgb_model.predict_proba(feat)
                        
                        ensemble = p_cnn * 0.6 + p_xgb * 0.4
                    except:
                        ensemble = p_cnn
                else:
                    ensemble = p_cnn
                
                ensemble = ensemble.flatten()
                top3_indices = np.argsort(ensemble)[-3:][::-1]
                
                results = []
                for i, idx in enumerate(top3_indices):
                    food_name = self.index_to_label[idx]
                    confidence = float(ensemble[idx])
                    results.append({
                        'rank': i + 1,
                        'food': food_name,
                        'confidence': confidence
                    })
                
                return results, "성공"
            
        except Exception as e:
            return None, f"예측 오류: {str(e)[:50]}"
    
    def load_properties_data(self, properties_file):
        if not properties_file or not os.path.exists(properties_file):
            self.log(f"속성 파일이 없습니다: {properties_file}", "WARNING")
            return
        
        try:
            self.log(f"속성 파일 로드 시작: {properties_file}", "INFO")
            excel_data = pd.ExcelFile(properties_file, engine='openpyxl')
            self.properties_data = {}
            
            total_menus = 0
            for sheet_name in excel_data.sheet_names:
                try:
                    df = pd.read_excel(properties_file, sheet_name=sheet_name, engine='openpyxl')
                    
                    df = df.dropna(how='all')
                    df = df.dropna(axis=1, how='all')
                    
                    if df.empty:
                        self.log(f"시트 '{sheet_name}' 비어있음 - 건너뜀", "WARNING")
                        continue
                    
                    # 컬럼명이 메뉴명인 구조로 변경
                    # 첫 번째 컬럼은 재료/속성명으로 가정
                    if len(df.columns) < 2:
                        self.log(f"시트 '{sheet_name}'에 충분한 컬럼이 없음 - 건너뜀", "WARNING")
                        continue
                    
                    # 첫 번째 컬럼을 인덱스로 사용 (재료명/속성명)
                    ingredient_col = df.columns[0]
                    df = df.set_index(ingredient_col)
                    
                    # 메뉴명들 (컬럼명들)
                    menu_names = [col for col in df.columns if not pd.isna(col) and str(col).strip()]
                    
                    if not menu_names:
                        self.log(f"시트 '{sheet_name}'에 유효한 메뉴명이 없음 - 건너뜀", "WARNING")
                        continue
                    
                    self.properties_data[sheet_name] = df
                    menu_count = len(menu_names)
                    total_menus += menu_count
                    
                    self.log(f"시트 로드 완료: '{sheet_name}' - {len(df)}개 재료/속성, {menu_count}개 메뉴", "SUCCESS")
                    
                    sample_menus = menu_names[:3]
                    self.log(f"  샘플 메뉴: {sample_menus}", "INFO")
                    
                except Exception as e:
                    self.log(f"시트 '{sheet_name}' 로드 실패: {e}", "ERROR")
                    continue
            
            if self.properties_data:
                self.log(f"속성 파일 로드 완료: {len(self.properties_data)}개 시트, 총 {total_menus}개 메뉴", "SUCCESS")
            else:
                self.log("속성 파일에서 유효한 데이터를 찾을 수 없음", "ERROR")
        
        except Exception as e:
            self.log(f"속성 파일 로드 실패: {e}", "ERROR")
    
    def get_menu_properties(self, menu_name):
        if not menu_name or not self.properties_data:
            return None
        
        menu_name = str(menu_name).strip()
        self.log(f"레시피 검색 시도: '{menu_name}'", "INFO")
        
        found_properties = {}
        found_menu_name = None
        
        for sheet_name, df in self.properties_data.items():
            # 컬럼명들에서 메뉴명 검색 (컬럼명이 메뉴명인 구조)
            menu_columns = [col for col in df.columns if not pd.isna(col) and str(col).strip()]
            
            # 1. 정확히 일치하는 메뉴 검색
            exact_match_col = None
            for col in menu_columns:
                if str(col).strip() == menu_name:
                    exact_match_col = col
                    found_menu_name = menu_name
                    self.log(f"정확 매칭 발견: '{menu_name}' in {sheet_name}", "SUCCESS")
                    break
            
            if exact_match_col:
                # 해당 컬럼의 모든 속성 추출 (빈 값과 '-' 제외)
                sheet_properties = {}
                for idx, value in df[exact_match_col].items():
                    if not pd.isna(value) and str(value).strip():
                        ingredient_name = str(idx).strip()
                        ingredient_value = str(value).strip()
                        # '-' 값과 빈 값 필터링
                        if (ingredient_name and ingredient_value and 
                            ingredient_value != 'nan' and 
                            ingredient_value != '-' and 
                            ingredient_value != ''):
                            sheet_properties[ingredient_name] = ingredient_value
                
                if sheet_properties:
                    found_properties[sheet_name] = sheet_properties
                    self.log(f"시트 '{sheet_name}'에서 {len(sheet_properties)}개 속성 발견", "SUCCESS")
                continue
            
            # 2. 부분 일치 검색 (컬럼명에 검색어가 포함)
            partial_match_col = None
            for col in menu_columns:
                if menu_name in str(col) or str(col) in menu_name:
                    partial_match_col = col
                    found_menu_name = str(col).strip()
                    self.log(f"부분 매칭 발견: '{menu_name}' → '{found_menu_name}' in {sheet_name}", "SUCCESS")
                    break
            
            if partial_match_col:
                # 해당 컬럼의 모든 속성 추출 (빈 값과 '-' 제외)
                sheet_properties = {}
                for idx, value in df[partial_match_col].items():
                    if not pd.isna(value) and str(value).strip():
                        ingredient_name = str(idx).strip()
                        ingredient_value = str(value).strip()
                        # '-' 값과 빈 값 필터링
                        if (ingredient_name and ingredient_value and 
                            ingredient_value != 'nan' and 
                            ingredient_value != '-' and 
                            ingredient_value != ''):
                            sheet_properties[ingredient_name] = ingredient_value
                
                if sheet_properties:
                    found_properties[sheet_name] = sheet_properties
                    self.log(f"시트 '{sheet_name}'에서 {len(sheet_properties)}개 속성 발견", "SUCCESS")
                continue
        
        if found_properties:
            result = {
                'menu_name': found_menu_name or menu_name,
                'properties': found_properties
            }
            self.log(f"레시피 검색 성공: '{menu_name}' → {len(found_properties)}개 시트에서 정보 발견", "SUCCESS")
            return result
        else:
            self.log(f"레시피 검색 실패: '{menu_name}' - 속성 파일에서 찾을 수 없음", "WARNING")
            # 디버깅을 위해 사용 가능한 메뉴명 샘플 출력
            if self.properties_data:
                sample_menus = []
                for sheet_name, df in self.properties_data.items():
                    menu_cols = [str(col).strip() for col in df.columns if not pd.isna(col) and str(col).strip()]
                    sample_menus.extend(menu_cols[:3])
                if sample_menus:
                    self.log(f"속성파일 메뉴 샘플: {sample_menus[:5]}", "INFO")
            return None

def cleanup_resources():
    try:
        gc.collect()
        tf.keras.backend.clear_session()
    except:
        pass

atexit.register(cleanup_resources)

class FoodPredictorGUI:
    def __init__(self):
        self._vars_created = False
        self._cleanup_done = False
        
        self.root = tk.Tk()
        self.root.title("음식 이미지 URL 연속 예측 시스템")
        self.root.geometry("1400x900")
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        self.predictor = None
        self.is_running = False
        self.stop_requested = False
        
        self.current_image = None
        self.current_results = None
        self.current_properties = None
        self.current_menu_name = None
        self.user_action = None
        
        self.log_queue = queue.Queue()
        self.ui_update_queue = queue.Queue()
        self.translation_table = {}
        
        self.setup_ui()
        self.load_translation_table()
        self.check_queues()
    
    def create_tkinter_vars(self):
        if self._vars_created:
            return
        
        try:
            self.folder_var = tk.StringVar(value="./excel_files")
            self.properties_var = tk.StringVar(value="./menu_properties.xlsx")
            
            self.show_images_var = tk.BooleanVar(value=True)
            self.auto_continue_var = tk.BooleanVar(value=False)
            self.fast_mode_var = tk.BooleanVar(value=False)
            self.parallel_mode_var = tk.BooleanVar(value=False)
            
            self.status_var = tk.StringVar(value="준비")
            self.total_files_var = tk.StringVar(value="파일: 0")
            self.total_processed_var = tk.StringVar(value="처리: 0")
            self.total_success_var = tk.StringVar(value="성공: 0")
            self.success_rate_var = tk.StringVar(value="성공률: 0%")
            self.elapsed_time_var = tk.StringVar(value="시간: 00:00")
            
            self.auto_scroll_var = tk.BooleanVar(value=True)
            self._vars_created = True
            
        except Exception as e:
            print(f"Tkinter 변수 생성 오류: {e}")
    
    def setup_ui(self):
        try:
            self.create_tkinter_vars()
            
            main_container = tk.Frame(self.root)
            main_container.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
            
            # 제목
            title_frame = tk.Frame(main_container)
            title_frame.pack(fill=tk.X, pady=(0, 10))
            
            title_label = tk.Label(title_frame, text="음식 이미지 URL 연속 예측 시스템", 
                                 font=('Arial', 16, 'bold'), fg='#2E86C1')
            title_label.pack()
            
            # 상단 영역
            top_frame = tk.Frame(main_container)
            top_frame.pack(fill=tk.X, pady=(0, 10))
            
            # 좌측 설정
            left_config = tk.LabelFrame(top_frame, text="폴더 및 파일 설정", font=('Arial', 11, 'bold'))
            left_config.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 5))
            
            folder_frame = tk.Frame(left_config)
            folder_frame.pack(fill=tk.X, padx=5, pady=3)
            tk.Label(folder_frame, text="Excel 폴더:", font=('Arial', 10)).pack(side=tk.LEFT)
            tk.Entry(folder_frame, textvariable=self.folder_var, width=40, font=('Arial', 9)).pack(side=tk.LEFT, padx=5, fill=tk.X, expand=True)
            tk.Button(folder_frame, text="찾기", command=self.browse_folder, font=('Arial', 9)).pack(side=tk.RIGHT)
            
            prop_frame = tk.Frame(left_config)
            prop_frame.pack(fill=tk.X, padx=5, pady=3)
            tk.Label(prop_frame, text="속성 파일:", font=('Arial', 10)).pack(side=tk.LEFT)
            tk.Entry(prop_frame, textvariable=self.properties_var, width=40, font=('Arial', 9)).pack(side=tk.LEFT, padx=5, fill=tk.X, expand=True)
            tk.Button(prop_frame, text="찾기", command=self.browse_properties, font=('Arial', 9)).pack(side=tk.RIGHT)
            
            options_frame = tk.Frame(left_config)
            options_frame.pack(fill=tk.X, padx=5, pady=3)
            
            tk.Checkbutton(options_frame, text="이미지 미리보기", variable=self.show_images_var, font=('Arial', 9)).pack(side=tk.LEFT)
            tk.Checkbutton(options_frame, text="자동 진행 (즉시)", variable=self.auto_continue_var, font=('Arial', 9)).pack(side=tk.LEFT, padx=10)
            tk.Checkbutton(options_frame, text="고속 처리 모드", variable=self.fast_mode_var, font=('Arial', 9)).pack(side=tk.LEFT)
            tk.Checkbutton(options_frame, text="병렬 처리 (10배 빠름)", variable=self.parallel_mode_var, font=('Arial', 9)).pack(side=tk.LEFT, padx=10)
            
            # 우측 로그
            log_frame = tk.LabelFrame(top_frame, text="실행 로그", font=('Arial', 11, 'bold'))
            log_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(5, 0))
            
            log_control = tk.Frame(log_frame)
            log_control.pack(fill=tk.X, padx=3, pady=2)
            
            tk.Button(log_control, text="지우기", command=self.clear_log, font=('Arial', 8)).pack(side=tk.LEFT)
            tk.Button(log_control, text="저장", command=self.save_log, font=('Arial', 8)).pack(side=tk.LEFT, padx=3)
            tk.Checkbutton(log_control, text="자동 스크롤", variable=self.auto_scroll_var, font=('Arial', 8)).pack(side=tk.RIGHT)
            
            self.log_text = scrolledtext.ScrolledText(log_frame, wrap=tk.WORD, font=('Consolas', 8), 
                                                     height=12, state=tk.DISABLED)
            self.log_text.pack(fill=tk.BOTH, expand=True, padx=3, pady=3)
            
            # 컨트롤 버튼
            control_frame = tk.Frame(main_container)
            control_frame.pack(fill=tk.X, pady=(0, 5))
            
            button_left = tk.Frame(control_frame)
            button_left.pack(side=tk.LEFT)
            
            self.start_button = tk.Button(button_left, text="시작", command=self.start_processing, 
                                        font=('Arial', 12, 'bold'), bg='#28a745', fg='white', padx=20)
            self.start_button.pack(side=tk.LEFT, padx=5)
            
            self.stop_button = tk.Button(button_left, text="중지", command=self.stop_processing, 
                                       font=('Arial', 12, 'bold'), bg='#dc3545', fg='white', 
                                       state=tk.DISABLED, padx=20)
            self.stop_button.pack(side=tk.LEFT, padx=5)
            
            self.continue_button = tk.Button(button_left, text="계속", command=self.user_continue, 
                                           font=('Arial', 10), bg='#007bff', fg='white', 
                                           state=tk.DISABLED, padx=15)
            self.continue_button.pack(side=tk.LEFT, padx=5)
            
            self.save_button = tk.Button(button_left, text="저장", command=self.user_save, 
                                       font=('Arial', 10), bg='#ffc107', fg='black', 
                                       state=tk.DISABLED, padx=15)
            self.save_button.pack(side=tk.LEFT, padx=5)
            
            # 통계
            stats_frame = tk.Frame(control_frame)
            stats_frame.pack(side=tk.RIGHT)
            
            tk.Label(stats_frame, textvariable=self.total_files_var, font=('Arial', 10)).pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.total_processed_var, font=('Arial', 10)).pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.total_success_var, font=('Arial', 10)).pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.success_rate_var, font=('Arial', 10)).pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.elapsed_time_var, font=('Arial', 10)).pack(side=tk.LEFT, padx=5)
            
            # 진행률
            progress_frame = tk.Frame(main_container)
            progress_frame.pack(fill=tk.X, pady=(0, 5))
            
            tk.Label(progress_frame, text="진행 상태:", font=('Arial', 10)).pack(side=tk.LEFT)
            tk.Label(progress_frame, textvariable=self.status_var, font=('Arial', 10), fg='blue').pack(side=tk.LEFT, padx=10)
            
            self.progress = ttk.Progressbar(progress_frame, mode='indeterminate')
            self.progress.pack(side=tk.RIGHT, fill=tk.X, expand=True, padx=10)
            
            # 메인 콘텐츠
            content_frame = tk.Frame(main_container)
            content_frame.pack(fill=tk.BOTH, expand=True)
            
            # 좌측 이미지
            image_panel = tk.LabelFrame(content_frame, text="이미지 분석", font=('Arial', 11, 'bold'))
            image_panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 5))
            
            self.image_frame = tk.Frame(image_panel)
            self.image_frame.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
            
            self.image_label = tk.Label(self.image_frame, text="이미지 분석을 시작하려면 '시작' 버튼을 눌러주세요", 
                                       font=('Arial', 10), fg='gray', 
                                       width=50, height=15, relief=tk.SUNKEN, bg='white')
            self.image_label.pack(expand=True, fill=tk.BOTH)
            
            image_info_frame = tk.Frame(image_panel)
            image_info_frame.pack(fill=tk.X, padx=5, pady=5)
            
            url_frame = tk.LabelFrame(image_info_frame, text="이미지 URL", font=('Arial', 9, 'bold'))
            url_frame.pack(fill=tk.X, pady=2)
            
            self.url_text = tk.Text(url_frame, height=2, font=('Arial', 8), state=tk.DISABLED)
            self.url_text.pack(fill=tk.X, padx=3, pady=2)
            
            result_frame = tk.LabelFrame(image_info_frame, text="예측 결과", font=('Arial', 9, 'bold'))
            result_frame.pack(fill=tk.X, pady=2)
            
            self.result_text = tk.Text(result_frame, height=3, font=('Arial', 9), state=tk.DISABLED)
            self.result_text.pack(fill=tk.X, padx=3, pady=2)
            
            # 우측 정보
            info_panel = tk.Frame(content_frame, width=400)
            info_panel.pack(side=tk.RIGHT, fill=tk.Y, padx=(5, 0))
            info_panel.pack_propagate(False)
            
            menu_frame = tk.LabelFrame(info_panel, text="메뉴 정보", font=('Arial', 11, 'bold'))
            menu_frame.pack(fill=tk.X, pady=(0, 5))
            
            self.menu_info_text = tk.Text(menu_frame, height=4, font=('Arial', 10), state=tk.DISABLED)
            self.menu_info_text.pack(fill=tk.X, padx=3, pady=3)
            
            recipe_frame = tk.LabelFrame(info_panel, text="레시피 정보", font=('Arial', 11, 'bold'))
            recipe_frame.pack(fill=tk.BOTH, expand=True)
            
            self.recipe_text = scrolledtext.ScrolledText(recipe_frame, wrap=tk.WORD, font=('Arial', 9), state=tk.DISABLED)
            self.recipe_text.pack(fill=tk.BOTH, expand=True, padx=3, pady=3)
            
            self.add_log("음식 이미지 예측 시스템이 준비되었습니다.", "INFO")
            self.add_log("Excel 폴더와 속성 파일 경로를 확인한 후 '시작' 버튼을 눌러주세요.", "INFO")
            self.test_network_connection()
            
        except Exception as e:
            print(f"UI 설정 오류: {e}")
    
    def on_closing(self):
        if self._cleanup_done:
            return
        
        try:
            self._cleanup_done = True
            self.stop_requested = True
            self.is_running = False
            
            if hasattr(self, 'processing_thread') and self.processing_thread and self.processing_thread.is_alive():
                self.processing_thread.join(timeout=2)
            
            if hasattr(self, 'image_label') and hasattr(self.image_label, 'image'):
                self.image_label.image = None
            
            self.current_image = None
            
            if self.predictor:
                self.predictor = None
            
            try:
                while not self.log_queue.empty():
                    self.log_queue.get_nowait()
                while not self.ui_update_queue.empty():
                    self.ui_update_queue.get_nowait()
            except:
                pass
            
            if self._vars_created:
                var_names = ['folder_var', 'properties_var', 'show_images_var', 'auto_continue_var', 
                           'fast_mode_var', 'parallel_mode_var', 'status_var', 'total_files_var',
                           'total_processed_var', 'total_success_var', 'success_rate_var', 
                           'elapsed_time_var', 'auto_scroll_var']
                
                for var_name in var_names:
                    try:
                        if hasattr(self, var_name):
                            delattr(self, var_name)
                    except:
                        pass
            
            cleanup_resources()
            self.root.quit()
            self.root.destroy()
            
        except Exception as e:
            print(f"종료 중 오류: {e}")
            try:
                self.root.destroy()
            except:
                pass
    
    def check_queues(self):
        if self._cleanup_done or self.stop_requested:
            return
        
        try:
            while not self.log_queue.empty():
                try:
                    message, level = self.log_queue.get_nowait()
                    self._add_log_to_ui(message, level)
                except queue.Empty:
                    break
                except:
                    break
            
            while not self.ui_update_queue.empty():
                try:
                    update_func, args = self.ui_update_queue.get_nowait()
                    update_func(*args)
                except queue.Empty:
                    break
                except:
                    break
            
            if not self.stop_requested and not self._cleanup_done:
                self.root.after(100, self.check_queues)
                
        except Exception as e:
            print(f"큐 확인 중 오류: {e}")
    
    def add_log(self, message, level="INFO"):
        if self._cleanup_done:
            return
        
        try:
            if threading.current_thread() == threading.main_thread():
                self._add_log_to_ui(message, level)
            else:
                self.log_queue.put((message, level))
        except Exception as e:
            print(f"로그 추가 오류: {e}")
    
    def _add_log_to_ui(self, message, level="INFO"):
        if self._cleanup_done:
            return
        
        try:
            if not hasattr(self, 'log_text') or not self.log_text.winfo_exists():
                return
            
            timestamp = datetime.now().strftime("%H:%M:%S")
            
            color_map = {
                "INFO": "#000000",
                "SUCCESS": "#008000",
                "WARNING": "#FF8C00",
                "ERROR": "#DC143C",
                "STEP": "#0000FF",
                "PROCESS": "#800080"
            }
            
            color = color_map.get(level, "#000000")
            
            self.log_text.config(state=tk.NORMAL)
            self.log_text.insert(tk.END, f"[{timestamp}] [{level}] {message}\n")
            
            line_start = self.log_text.index("end-2l linestart")  
            line_end = self.log_text.index("end-1l lineend")
            tag_name = f"{level}_{timestamp}"
            self.log_text.tag_add(tag_name, line_start, line_end)
            self.log_text.tag_config(tag_name, foreground=color)
            
            self.log_text.config(state=tk.DISABLED)
            
            if hasattr(self, 'auto_scroll_var') and self.auto_scroll_var.get():
                self.log_text.see(tk.END)
            
        except Exception as e:
            print(f"UI 로그 업데이트 오류: {e}")
    
    def clear_log(self):
        try:
            self.log_text.config(state=tk.NORMAL)
            self.log_text.delete(1.0, tk.END)
            self.log_text.config(state=tk.DISABLED)
        except Exception as e:
            print(f"로그 지우기 오류: {e}")
    
    def save_log(self):
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"log_{timestamp}.txt"
            
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(self.log_text.get(1.0, tk.END))
            
            self.add_log(f"로그가 저장되었습니다: {filename}", "SUCCESS")
            
        except Exception as e:
            self.add_log(f"로그 저장 실패: {e}", "ERROR")
    
    def test_network_connection(self):
        test_urls = [
            "https://httpbin.org/image/jpeg",
            "https://via.placeholder.com/300x300.jpg",
            "https://picsum.photos/200/200"
        ]
        
        self.add_log("네트워크 연결 테스트 중...", "INFO")
        
        success_count = 0
        for i, test_url in enumerate(test_urls):
            try:
                img, msg = self.predictor.download_image(test_url) if self.predictor else (None, "예측기 미초기화")
                if img is not None:
                    success_count += 1
                    self.add_log(f"테스트 {i+1}/3 성공: {test_url[:50]}...", "SUCCESS")
                    break
                else:
                    self.add_log(f"테스트 {i+1}/3 실패: {msg}", "WARNING")
            except Exception as e:
                self.add_log(f"테스트 {i+1}/3 오류: {str(e)[:50]}", "ERROR")
        
        if success_count > 0:
            self.add_log("네트워크 연결 정상 - 이미지 다운로드 가능", "SUCCESS")
        else:
            self.add_log("네트워크 연결 문제 - 방화벽/프록시 설정 확인 필요", "ERROR")
            self.add_log("해결 방법: 1) 인터넷 연결 확인 2) 방화벽 설정 3) 프록시 설정", "INFO")
    
    def browse_folder(self):
        try:
            folder = filedialog.askdirectory(title="Excel 파일이 있는 폴더를 선택하세요")
            if folder:
                self.folder_var.set(folder)
        except Exception as e:
            self.add_log(f"폴더 선택 오류: {e}", "ERROR")
    
    def browse_properties(self):
        try:
            file = filedialog.askopenfilename(
                title="메뉴 속성 Excel 파일을 선택하세요",
                filetypes=[("Excel files", "*.xlsx *.xls"), ("All files", "*.*")]
            )
            if file:
                self.properties_var.set(file)
        except Exception as e:
            self.add_log(f"속성 파일 선택 오류: {e}", "ERROR")
    
    def load_translation_table(self):
        try:
            self.translation_table = {
                'BBQ': '바비큐립',
                'baguette': '바게트',
                'banh_mi': '반미',
                'bingsu': '팥빙수',
                'bulgogi': '불고기',
                'bunza': '분짜',
                'burger': '치즈버거',
                'burrito': '부리또',
                'cake': '치즈케이크',
                'chicken': '후라이드치킨',
                'cookie': '쿠키',
                'croissant': '크루아상',
                'croque_monsieur': '크로크무슈',
                'curry': '레드커리',
                'dim_sum': '딤섬',
                'egg_benedict': '에그베네딕트',
                'french_fries': '감자튀김',
                'french_toast': '프렌치토스트',
                'galbi': '돼지갈비',
                'gimbap': '참치김밥',
                'gratin': '그라탱',
                'hot_pot': '훠궈',
                'jajangmyeon': '짜장면',
                'japchae': '잡채밥',
                'kebap': '케밥',
                'kimchi_stew': '김치찌개',
                'korean_pancake': '해물전',
                'lasana': '라자냐',
                'macaroon': '마카롱',
                'mapa_tofu': '마파두부밥',
                'muffin': '머핀',
                'nachos': '나초',
                'pad_thai': '팟타이',
                'pan_cake': '팬케이크',
                'pasta': '마라파스타',
                'pizza': '한국식피자',
                'quesadilla': '케사디아',
                'ramen': '라멘',
                'rice_noodle': '쌀국수',
                'risotto': '해물리조또',
                'salad': '시저샐러드',
                'sashimi': '생선회',
                'seaweed_soup': '미역국',
                'soba': '소바',
                'soup': '크림수프',
                'steak': '스테이크와감자',
                'sushi': '연어초밥',
                'takoyaki': '타코야키',
                'tteokbokki': '국물떡볶이',
                'udon': '우동'
            }
            
            self.add_log(f"음식 매핑 테이블 로드 완료: {len(self.translation_table)}개", "SUCCESS")
            
        except Exception as e:
            self.add_log(f"번역 테이블 로드 실패: {e}", "ERROR")
    
    def translate_food_name(self, english_name):
        return self.translation_table.get(english_name, english_name)
    
    def get_recipe_search_keyword(self, results, sheet_menu_name):
        search_keywords = []
        
        if results:
            english_prediction = results[0]['food']
            
            korean_mapped = self.translation_table.get(english_prediction)
            if korean_mapped:
                search_keywords.append({
                    'keyword': korean_mapped,
                    'source': 'AI예측(한글매핑)',
                    'confidence': results[0]['confidence']
                })
            else:
                search_keywords.append({
                    'keyword': english_prediction,
                    'source': 'AI예측(매핑없음)',
                    'confidence': results[0]['confidence']
                })
        
        return search_keywords
    
    def create_file_summary(self, all_sheets_data, file_name):
        """각 시트별 처리 결과 요약 생성"""
        try:
            summary_rows = []
            
            for sheet_name, df in all_sheets_data.items():
                # 예측 결과가 있는 시트만 처리
                if '예측_음식명' not in df.columns:
                    continue
                    
                # 각 시트별 통계 계산
                total_rows = len(df)
                
                # URL이 있는 행 수 (실제 처리 대상)
                url_cols = [col for col in df.columns if 'url' in str(col).lower() or 'URL' in str(col)]
                actual_urls = 0
                if url_cols:
                    actual_urls = df[url_cols[0]].notna().sum()
                
                # 예측이 시도된 행 수
                predicted_rows = df['예측_음식명'].notna().sum()
                
                # 성공한 예측 수 (오류가 아닌 것들)
                error_conditions = df['예측_음식명'].str.contains('오류|예측오류', na=False)
                success_rows = df[df['예측_음식명'].notna() & ~error_conditions]['예측_음식명'].count()
                
                # 예측된 음식명 빈도 (상위 3개)
                valid_predictions = df[df['예측_음식명'].notna() & ~error_conditions]['예측_음식명']
                predictions = valid_predictions.value_counts().head(3)
                
                # 평균 신뢰도 계산
                confidence_col = '예측_신뢰도'
                avg_confidence = 0
                if confidence_col in df.columns:
                    confidence_values = pd.to_numeric(df[confidence_col], errors='coerce')
                    avg_confidence = confidence_values.mean() if not confidence_values.isna().all() else 0
                
                summary_row = {
                    '파일명': file_name,
                    '시트명': sheet_name,
                    '전체_행수': total_rows,
                    'URL_개수': actual_urls,
                    '처리_시도수': predicted_rows,
                    '처리_성공수': success_rows,
                    '성공률(%)': f"{(success_rows/actual_urls*100):.1f}%" if actual_urls > 0 else "0%",
                    '평균_신뢰도': f"{avg_confidence:.3f}" if avg_confidence > 0 else "0.000",
                    '최다_예측음식': predictions.index[0] if len(predictions) > 0 else "",
                    '최다_예측빈도': predictions.iloc[0] if len(predictions) > 0 else 0,
                    '2위_예측음식': predictions.index[1] if len(predictions) > 1 else "",
                    '2위_예측빈도': predictions.iloc[1] if len(predictions) > 1 else 0,
                    '3위_예측음식': predictions.index[2] if len(predictions) > 2 else "",
                    '3위_예측빈도': predictions.iloc[2] if len(predictions) > 2 else 0
                }
                summary_rows.append(summary_row)
            
            if summary_rows:
                summary_df = pd.DataFrame(summary_rows)
                self.add_log(f"요약 데이터 생성 완료: {len(summary_rows)}개 시트", "SUCCESS")
                return summary_df
            else:
                self.add_log("요약할 데이터가 없습니다", "WARNING")
                return None
                
        except Exception as e:
            self.add_log(f"요약 생성 오류: {e}", "ERROR")
            return None
    
    def start_processing(self):
        if self.is_running:
            return
        
        folder_path = self.folder_var.get()
        properties_file = self.properties_var.get()
        
        if not os.path.exists(folder_path):
            messagebox.showerror("오류", f"Excel 폴더가 존재하지 않습니다:\n{folder_path}")
            return
        
        if not os.path.exists('models'):
            messagebox.showerror("오류", "models 폴더가 존재하지 않습니다.")
            return
        
        self.is_running = True
        self.stop_requested = False  
        self.user_action = None
        
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        
        self.progress.config(mode='indeterminate')
        self.progress.start()
        
        self.processing_thread = threading.Thread(target=self.run_processing, daemon=True)
        self.processing_thread.start()
    
    def stop_processing(self):
        self.stop_requested = True
        self.add_log("중지 요청됨 - 현재 작업 완료 후 중단됩니다.", "WARNING")
    
    def user_continue(self):
        self.user_action = 'continue'
    
    def user_save(self):
        self.user_action = 'save'
    
    def wait_for_user_action(self):
        if self.auto_continue_var.get():
            return 'continue'
        
        self.continue_button.config(state=tk.NORMAL)
        self.save_button.config(state=tk.NORMAL)
        
        self.user_action = None
        while self.user_action is None and not self.stop_requested:
            time.sleep(0.1)
        
        self.continue_button.config(state=tk.DISABLED)
        self.save_button.config(state=tk.DISABLED)
        
        return self.user_action or 'quit'
    
    def add_ai_analysis_to_summary(self, summary_df, file_menu_stats, file_name):
        """기존 요약 시트에 AI 예측 분석 결과를 가로로 추가"""
        try:
            # 기존 요약 데이터 복사
            updated_summary = summary_df.copy()
            
            # AI 분석 결과 컬럼들 추가
            ai_columns = [
                'AI_전체이미지수',
                'AI_정확예측수', 
                'AI_정확률(%)',
                'AI_오예측1위',
                'AI_오예측1위_빈도',
                'AI_오예측2위',
                'AI_오예측2위_빈도',
                'AI_오예측3위',
                'AI_오예측3위_빈도'
            ]
            
            # 새 컬럼들을 빈 값으로 초기화
            for col in ai_columns:
                updated_summary[col] = ""
            
            # 메뉴명 컬럼 찾기 - 요약시트에서 메뉴명이 있는 컬럼
            menu_column = None
            possible_menu_columns = ['메뉴', '메뉴명', '음식명', '음식', '메뉴이름']
            
            # 첫 번째 컬럼이 메뉴명일 가능성이 높음
            if len(updated_summary.columns) > 0:
                first_col = updated_summary.columns[0]
                if any(keyword in str(first_col) for keyword in possible_menu_columns):
                    menu_column = first_col
                else:
                    # 첫 번째 컬럼이 메뉴명이 아니라면 다른 컬럼에서 찾기
                    for col in updated_summary.columns:
                        col_str = str(col).lower()
                        if any(keyword in col_str for keyword in possible_menu_columns):
                            menu_column = col
                            break
                    
                    # 그래도 못 찾으면 첫 번째 컬럼을 메뉴명으로 가정
                    if menu_column is None:
                        menu_column = first_col
                        self.add_log(f"메뉴명 컬럼을 찾지 못해 첫 번째 컬럼 '{first_col}'을 사용", "WARNING")
            
            if menu_column is None:
                self.add_log("요약 시트에서 메뉴명 컬럼을 찾을 수 없음", "WARNING")
                return updated_summary
            
            self.add_log(f"요약시트 메뉴명 컬럼: '{menu_column}'", "INFO")
            
            # 매칭 통계
            matched_count = 0
            total_rows = 0
            
            # 각 메뉴별로 AI 분석 결과 매칭하여 추가
            for idx, row in updated_summary.iterrows():
                menu_name_in_summary = str(row[menu_column]).strip()
                
                # 빈 값이나 헤더가 아닌 경우만 처리
                if menu_name_in_summary and menu_name_in_summary != 'nan' and len(menu_name_in_summary) > 0:
                    total_rows += 1
                    
                    # file_menu_stats에서 매칭되는 메뉴 찾기
                    found_stats = None
                    matched_menu_key = None
                    
                    # 1. 정확히 일치하는 메뉴 찾기
                    if menu_name_in_summary in file_menu_stats:
                        found_stats = file_menu_stats[menu_name_in_summary]
                        matched_menu_key = menu_name_in_summary
                    else:
                        # 2. 부분 일치 검색 (시트명이 메뉴명을 포함하는 경우)
                        for stats_key in file_menu_stats.keys():
                            if (menu_name_in_summary in stats_key or 
                                stats_key in menu_name_in_summary or
                                menu_name_in_summary.replace(' ', '') == stats_key.replace(' ', '')):
                                found_stats = file_menu_stats[stats_key]
                                matched_menu_key = stats_key
                                break
                    
                    if found_stats:
                        matched_count += 1
                        total = found_stats['total']
                        correct = found_stats['correct']
                        accuracy = (correct / total * 100) if total > 0 else 0
                        wrong_predictions = found_stats['wrong_predictions']
                        
                        # 기본 통계 추가
                        updated_summary.loc[idx, 'AI_전체이미지수'] = total
                        updated_summary.loc[idx, 'AI_정확예측수'] = correct
                        updated_summary.loc[idx, 'AI_정확률(%)'] = f"{accuracy:.1f}%"
                        
                        # 오예측 상위 3개 추가
                        if wrong_predictions:
                            sorted_wrong = sorted(wrong_predictions.items(), key=lambda x: x[1], reverse=True)
                            
                            for i, (wrong_pred, count) in enumerate(sorted_wrong[:3]):
                                if i == 0:
                                    updated_summary.loc[idx, 'AI_오예측1위'] = wrong_pred
                                    updated_summary.loc[idx, 'AI_오예측1위_빈도'] = count
                                elif i == 1:
                                    updated_summary.loc[idx, 'AI_오예측2위'] = wrong_pred
                                    updated_summary.loc[idx, 'AI_오예측2위_빈도'] = count
                                elif i == 2:
                                    updated_summary.loc[idx, 'AI_오예측3위'] = wrong_pred
                                    updated_summary.loc[idx, 'AI_오예측3위_빈도'] = count
                        else:
                            # 오예측이 없는 경우
                            updated_summary.loc[idx, 'AI_오예측1위'] = "(정확예측)"
                            updated_summary.loc[idx, 'AI_오예측1위_빈도'] = 0
                        
                        self.add_log(f"매칭 성공: 요약시트 '{menu_name_in_summary}' ↔ 시트명 '{matched_menu_key}' (정확도: {accuracy:.1f}%)", "SUCCESS")
                    else:
                        self.add_log(f"매칭 실패: 요약시트 '{menu_name_in_summary}' - 해당 시트의 AI 분석 결과 없음", "WARNING")
            
            self.add_log(f"요약시트 AI 분석 컬럼 추가 완료: {matched_count}/{total_rows} 메뉴 매칭됨", "SUCCESS")
            
            # 디버깅 정보
            if matched_count == 0:
                self.add_log("매칭된 메뉴가 없습니다. 디버깅 정보:", "WARNING")
                summary_menus = [str(row[menu_column]).strip() for _, row in updated_summary.iterrows() if str(row[menu_column]).strip()]
                stats_keys = list(file_menu_stats.keys())
                self.add_log(f"요약시트 메뉴들: {summary_menus[:5]}", "INFO")
                self.add_log(f"분석결과 키들: {stats_keys[:5]}", "INFO")
            
            return updated_summary
            
        except Exception as e:
            self.add_log(f"요약 시트 업데이트 오류: {e}", "ERROR")
            return summary_df  # 오류 시 원본 반환
    
    def run_processing(self):
        try:
            self.add_log("예측 모델 로드 중...", "STEP")
            self.predictor = FoodImagePredictor()
            self.predictor.load_models()
            
            properties_file = self.properties_var.get()
            if properties_file and os.path.exists(properties_file):
                self.predictor.load_properties_data(properties_file)
            
            self.add_log("모든 모델이 성공적으로 로드되었습니다!", "SUCCESS")
            
            folder_path = self.folder_var.get()
            excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))
            excel_files = [f for f in excel_files if not os.path.basename(f).startswith('~')]
            
            if not excel_files:
                self.add_log(f"Excel 파일을 찾을 수 없습니다: {folder_path}", "ERROR")
                return
            
            # 결과 폴더 생성
            folder_name = os.path.basename(folder_path.rstrip('/\\'))
            result_folder = f"{folder_path}_결과"
            if not os.path.exists(result_folder):
                os.makedirs(result_folder)
                self.add_log(f"결과 폴더 생성: {result_folder}", "SUCCESS")
            else:
                self.add_log(f"결과 폴더 사용: {result_folder}", "INFO")
            
            self.add_log(f"Excel 파일 {len(excel_files)}개 발견", "SUCCESS")
            
            total_processed = 0
            total_success = 0
            start_time = time.time()
            
            # 요약 통계를 위한 변수들
            menu_match_count = 0  # 메뉴명과 예측이 일치한 개수
            prediction_stats = {}  # 예측된 음식명별 빈도
            
            for file_idx, excel_file in enumerate(excel_files):
                if self.stop_requested:
                    break
                
                file_name = os.path.basename(excel_file)
                result_file_path = os.path.join(result_folder, file_name)
                
                self.add_log(f"파일 처리 시작: {file_name}", "STEP")
                self.add_log(f"결과 저장 경로: {result_file_path}", "INFO")
                
                try:
                    excel_data = pd.ExcelFile(excel_file, engine='openpyxl')
                    all_sheets_data = {}  # 모든 시트 데이터를 저장
                    
                    # 파일별 통계 변수
                    file_menu_stats = {}  # 메뉴별 정확/오예측 통계
                    
                    for sheet_name in excel_data.sheet_names:
                        if self.stop_requested:
                            break
                        
                        df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl')
                        
                        # 요약 시트는 URL 처리하지 않고 원본 그대로 보관
                        if sheet_name == '요약':
                            all_sheets_data[sheet_name] = df
                            self.add_log(f"요약 시트 원본 보관: {sheet_name}", "INFO")
                            continue
                        
                        url_col = None
                        # URL 컬럼 찾기 - 실제 URL이 있는 컬럼을 찾아야 함
                        for col in df.columns:
                            col_str = str(col).lower()
                            # URL수, URL개수 같은 숫자 컬럼은 제외
                            if ('url' in col_str or 'URL' in str(col)) and '수' not in col_str and '개수' not in col_str and '갯수' not in col_str:
                                # 해당 컬럼에 실제 URL이 있는지 확인
                                sample_values = df[col].dropna().head(3)
                                if len(sample_values) > 0:
                                    # 샘플 값들이 실제 URL 형태인지 확인
                                    url_like_count = 0
                                    for val in sample_values:
                                        val_str = str(val).strip()
                                        if val_str.startswith(('http://', 'https://')) and '.' in val_str:
                                            url_like_count += 1
                                    
                                    # 샘플의 절반 이상이 URL 형태라면 이 컬럼을 선택
                                    if url_like_count >= len(sample_values) / 2:
                                        url_col = col
                                        self.add_log(f"URL 컬럼 발견: '{col}' (샘플: {url_like_count}/{len(sample_values)}개 URL 형태)", "SUCCESS")
                                        break
                        
                        if url_col is None:
                            self.add_log(f"URL 컬럼을 찾을 수 없습니다: {sheet_name}", "WARNING")
                            all_sheets_data[sheet_name] = df  # 원본 데이터 그대로 저장
                            continue
                        
                        menu_col = None
                        for col in df.columns:
                            if '메뉴' in str(col) or 'menu' in str(col).lower():
                                menu_col = col
                                break
                        
                        valid_urls = df[url_col].notna().sum()
                        if valid_urls == 0:
                            all_sheets_data[sheet_name] = df  # 원본 데이터 그대로 저장
                            continue
                        
                        self.add_log(f"시트 처리: {sheet_name} ({valid_urls}개 URL)", "STEP")
                        
                        # 새로운 결과 컬럼들 추가 준비
                        if '예측_음식명' not in df.columns:
                            df['예측_음식명'] = ""
                        if '예측_신뢰도' not in df.columns:
                            df['예측_신뢰도'] = ""
                        if '예측_2위' not in df.columns:
                            df['예측_2위'] = ""
                        if '예측_3위' not in df.columns:
                            df['예측_3위'] = ""
                        if '예측_영어명' not in df.columns:
                            df['예측_영어명'] = ""
                        if '예측_영어_2위' not in df.columns:
                            df['예측_영어_2위'] = ""
                        if '예측_영어_3위' not in df.columns:
                            df['예측_영어_3위'] = ""
                        if '레시피_검색키워드' not in df.columns:
                            df['레시피_검색키워드'] = ""
                        if '레시피_매칭결과' not in df.columns:
                            df['레시피_매칭결과'] = ""
                        if '레시피_시트수' not in df.columns:
                            df['레시피_시트수'] = ""
                        
                        sheet_processed = 0
                        sheet_success = 0
                        
                        for idx, row in df.iterrows():
                            if self.stop_requested:
                                break
                            
                            url = row[url_col]
                            if pd.isna(url) or str(url).strip() == '':
                                continue
                            
                            sheet_menu_name = None
                            if menu_col and not pd.isna(row[menu_col]):
                                sheet_menu_name = str(row[menu_col]).strip()
                            
                            sheet_processed += 1
                            total_processed += 1
                            
                            progress_info = f"[{sheet_processed}/{valid_urls}] {sheet_name}"
                            self.safe_ui_update(lambda info: self.status_var.set(f"처리 중: {info}"), progress_info)
                            
                            progress_percent = (sheet_processed / valid_urls) * 100
                            self.safe_ui_update(lambda p: self.progress.config(mode='determinate', value=p), progress_percent)
                            
                            self.add_log(f"[{sheet_processed:3d}/{valid_urls}] 이미지 분석 중...", "PROCESS")
                            if sheet_menu_name:
                                self.add_log(f"    시트 메뉴명: {sheet_menu_name}", "PROCESS")
                            
                            pil_image, download_msg = self.predictor.download_image(url)
                            
                            if pil_image is None:
                                self.add_log(f"다운로드 실패: {download_msg}", "WARNING")
                                # 실패한 경우에도 실패 정보 기록
                                df.loc[idx, '예측_음식명'] = f"오류: {download_msg}"
                                continue
                            
                            results, predict_msg = self.predictor.predict_image(pil_image)
                            
                            if results is None:
                                self.add_log(f"예측 실패: {predict_msg}", "WARNING")
                                df.loc[idx, '예측_음식명'] = f"예측오류: {predict_msg}"
                                continue
                            
                            sheet_success += 1
                            total_success += 1
                            
                            english_prediction = results[0]['food']
                            korean_prediction = self.translate_food_name(english_prediction)
                            
                            # 파일별 메뉴 통계 업데이트
                            if sheet_menu_name:
                                if sheet_menu_name not in file_menu_stats:
                                    file_menu_stats[sheet_menu_name] = {
                                        'total': 0,
                                        'correct': 0,
                                        'wrong_predictions': {}
                                    }
                                
                                file_menu_stats[sheet_menu_name]['total'] += 1
                                
                                # 정확 예측 여부 확인
                                sheet_menu_clean = sheet_menu_name.lower().strip()
                                korean_pred_clean = korean_prediction.lower().strip()
                                
                                if (sheet_menu_clean == korean_pred_clean or 
                                    sheet_menu_clean in korean_pred_clean or 
                                    korean_pred_clean in sheet_menu_clean):
                                    file_menu_stats[sheet_menu_name]['correct'] += 1
                                    menu_match_count += 1
                                    self.add_log(f"메뉴명 일치! 시트메뉴: '{sheet_menu_name}' ↔ 예측: '{korean_prediction}'", "SUCCESS")
                                else:
                                    # 오예측 카운트
                                    wrong_preds = file_menu_stats[sheet_menu_name]['wrong_predictions']
                                    if korean_prediction in wrong_preds:
                                        wrong_preds[korean_prediction] += 1
                                    else:
                                        wrong_preds[korean_prediction] = 1
                            
                            # 예측 통계 업데이트
                            if korean_prediction in prediction_stats:
                                prediction_stats[korean_prediction] += 1
                            else:
                                prediction_stats[korean_prediction] = 1
                            
                            # 예측 결과 저장
                            df.loc[idx, '예측_음식명'] = korean_prediction
                            df.loc[idx, '예측_신뢰도'] = round(results[0]['confidence'], 3)
                            if len(results) > 1:
                                df.loc[idx, '예측_2위'] = self.translate_food_name(results[1]['food'])
                            if len(results) > 2:
                                df.loc[idx, '예측_3위'] = self.translate_food_name(results[2]['food'])
                            
                            df.loc[idx, '예측_영어명'] = english_prediction
                            if len(results) > 1:
                                df.loc[idx, '예측_영어_2위'] = results[1]['food']
                            if len(results) > 2:
                                df.loc[idx, '예측_영어_3위'] = results[2]['food']
                            
                            self.add_log(f"예측 성공: {korean_prediction} ({english_prediction}) (신뢰도: {results[0]['confidence']:.3f})", "SUCCESS")
                            
                            # 레시피 검색 및 결과 저장
                            search_keywords = self.get_recipe_search_keyword(results, sheet_menu_name)
                            properties_info = None
                            search_result_info = None
                            
                            if search_keywords:
                                keyword_info = search_keywords[0]
                                keyword = keyword_info['keyword']
                                source = keyword_info['source']
                                
                                df.loc[idx, '레시피_검색키워드'] = f"{keyword} ({source})"
                                
                                self.add_log(f"레시피 검색 시도: '{keyword}' ({source})", "PROCESS")
                                properties_info = self.predictor.get_menu_properties(keyword)
                                
                                if properties_info:
                                    search_result_info = {
                                        'used_keyword': keyword,
                                        'keyword_source': source,
                                        'found_menu': properties_info['menu_name'],
                                        'found_sheets': list(properties_info['properties'].keys())
                                    }
                                    
                                    df.loc[idx, '레시피_매칭결과'] = properties_info['menu_name']
                                    df.loc[idx, '레시피_시트수'] = len(properties_info['properties'])
                                    
                                    self.add_log(f"레시피 검색 성공: '{keyword}' ({source}) → {properties_info['menu_name']} ({len(properties_info['properties'])}개 시트)", "SUCCESS")
                                else:
                                    df.loc[idx, '레시피_매칭결과'] = "매칭실패"
                                    df.loc[idx, '레시피_시트수'] = 0
                                    
                                    self.add_log(f"레시피 검색 실패: '{keyword}' ({source})", "WARNING")
                                    if self.predictor.properties_data:
                                        sample_menus = []
                                        for sheet_name_inner, df_inner in self.predictor.properties_data.items():
                                            menu_cols = [str(col).strip() for col in df_inner.columns if not pd.isna(col) and str(col).strip()]
                                            sample_menus.extend(menu_cols[:3])
                                        if sample_menus:
                                            self.add_log(f"속성파일 메뉴 샘플: {sample_menus[:5]}", "INFO")
                            else:
                                df.loc[idx, '레시피_검색키워드'] = "키워드생성실패"
                                df.loc[idx, '레시피_매칭결과'] = "검색실패"
                                df.loc[idx, '레시피_시트수'] = 0
                                self.add_log("검색 키워드 생성 실패", "ERROR")
                            
                            self.current_image = pil_image
                            self.current_results = results
                            self.current_properties = properties_info
                            self.current_menu_name = sheet_menu_name
                            self.current_search_info = search_result_info
                            
                            self.safe_ui_update(self.update_image_display, pil_image, url, results, sheet_menu_name, properties_info, search_result_info)
                            self.update_stats_safe(len(excel_files), total_processed, total_success, start_time)
                            
                            if self.show_images_var.get() and not self.fast_mode_var.get():
                                user_action = self.wait_for_user_action()
                                
                                if user_action == 'quit':
                                    self.stop_requested = True
                                    break
                                elif user_action == 'save':
                                    self.add_log("이미지 저장됨", "INFO")
                        
                        # 처리된 시트 데이터 저장
                        all_sheets_data[sheet_name] = df
                        self.add_log(f"시트 '{sheet_name}' 처리 완료: {sheet_success}/{sheet_processed} 성공", "SUCCESS")
                    
                    # 기존 요약 시트에 AI 예측 분석 결과 추가
                    if '요약' in all_sheets_data and file_menu_stats:
                        summary_df = all_sheets_data['요약']
                        updated_summary = self.add_ai_analysis_to_summary(summary_df, file_menu_stats, file_name)
                        all_sheets_data['요약'] = updated_summary
                        self.add_log(f"기존 요약 시트에 AI 분석 결과 추가됨", "SUCCESS")
                    
                    # 결과 파일 저장 (모든 시트 포함)
                    with pd.ExcelWriter(result_file_path, engine='openpyxl') as writer:
                        # 기존 시트들 저장
                        for sheet_name, sheet_df in all_sheets_data.items():
                            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
                        
                        # 새로운 요약 시트 생성 (기존 요약 시트가 없는 경우에만)
                        if '요약' not in all_sheets_data:
                            summary_data = self.create_file_summary(all_sheets_data, file_name)
                            if summary_data is not None:
                                summary_data.to_excel(writer, sheet_name='요약', index=False)
                                self.add_log(f"새 요약 시트 생성됨: {len(summary_data)}개 항목", "SUCCESS")
                    
                    self.add_log(f"결과 파일 저장 완료: {file_name}", "SUCCESS")
                    
                except Exception as e:
                    self.add_log(f"파일 처리 오류 ({file_name}): {e}", "ERROR")
                    continue
            
            self.update_stats_safe(len(excel_files), total_processed, total_success, start_time)
            
            # 요약 결과 출력
            self.add_log("=" * 60, "STEP")
            self.add_log("처리 요약 결과", "STEP")
            self.add_log("=" * 60, "STEP")
            
            if total_success > 0:
                self.add_log(f"처리 완료! 총 {total_success}개 이미지 분석 성공", "SUCCESS")
                self.add_log(f"결과 파일들이 다음 폴더에 저장되었습니다: {result_folder}", "SUCCESS")
                
                # 메뉴명 일치 통계
                if menu_match_count > 0:
                    match_rate = (menu_match_count / total_success) * 100
                    self.add_log(f"메뉴명 일치: {menu_match_count}개 ({match_rate:.1f}%)", "SUCCESS")
                else:
                    self.add_log("메뉴명 일치: 0개 (0%)", "WARNING")
                
                # 예측된 음식명 빈도 순 출력
                if prediction_stats:
                    self.add_log("예측된 음식명 빈도 (상위 10개):", "STEP")
                    sorted_predictions = sorted(prediction_stats.items(), key=lambda x: x[1], reverse=True)
                    
                    for i, (food_name, count) in enumerate(sorted_predictions[:10]):
                        percentage = (count / total_success) * 100
                        self.add_log(f"  {i+1:2d}. {food_name}: {count}개 ({percentage:.1f}%)", "INFO")
                    
                    if len(sorted_predictions) > 10:
                        remaining_count = sum(count for _, count in sorted_predictions[10:])
                        remaining_types = len(sorted_predictions) - 10
                        self.add_log(f"  기타 {remaining_types}개 음식: {remaining_count}개", "INFO")
                
                self.add_log("=" * 60, "STEP")
            else:
                self.add_log("처리된 이미지가 없습니다.", "WARNING")
        
        except Exception as e:
            self.add_log(f"처리 중 오류 발생: {e}", "ERROR")
            try:
                self.root.after(0, lambda: messagebox.showerror("오류", f"처리 중 오류가 발생했습니다:\n{e}"))
            except:
                pass
        finally:
            self.root.after(0, self.finish_processing)
    
    def finish_processing(self):
        try:
            self.is_running = False
            self.start_button.config(state=tk.NORMAL)
            self.stop_button.config(state=tk.DISABLED)
            self.continue_button.config(state=tk.DISABLED)
            self.save_button.config(state=tk.DISABLED)
            self.progress.stop()
            self.progress.config(mode='determinate', value=0)
            
            if self.stop_requested:
                self.status_var.set("사용자 중지")
                self.add_log("사용자 요청으로 처리가 중단되었습니다.", "WARNING")
            else:
                self.status_var.set("처리 완료")
                self.add_log("모든 처리가 완료되었습니다!", "SUCCESS")
                
        except Exception as e:
            print(f"처리 완료 정리 중 오류: {e}")
    
    def safe_ui_update(self, update_func, *args):
        try:
            if threading.current_thread() == threading.main_thread():
                update_func(*args)
            else:
                self.ui_update_queue.put((update_func, args))
        except Exception as e:
            print(f"UI 업데이트 오류: {e}")
    
    def update_stats_safe(self, total_files=0, total_processed=0, total_success=0, start_time=None):
        def _update():
            try:
                if hasattr(self, 'total_files_var'):
                    self.total_files_var.set(f"파일: {total_files}")
                if hasattr(self, 'total_processed_var'):
                    self.total_processed_var.set(f"처리: {total_processed}")
                if hasattr(self, 'total_success_var'):
                    self.total_success_var.set(f"성공: {total_success}")
                
                if total_processed > 0:
                    success_rate = (total_success / total_processed) * 100
                    if hasattr(self, 'success_rate_var'):
                        self.success_rate_var.set(f"성공률: {success_rate:.1f}%")
                else:
                    if hasattr(self, 'success_rate_var'):
                        self.success_rate_var.set("성공률: 0%")
                
                if start_time and hasattr(self, 'elapsed_time_var'):
                    elapsed = time.time() - start_time
                    hours = int(elapsed // 3600)
                    minutes = int((elapsed % 3600) // 60)
                    seconds = int(elapsed % 60)
                    if hours > 0:
                        time_str = f"시간: {hours:02d}:{minutes:02d}:{seconds:02d}"
                    else:
                        time_str = f"시간: {minutes:02d}:{seconds:02d}"
                    self.elapsed_time_var.set(time_str)
            except Exception as e:
                print(f"통계 업데이트 오류: {e}")
        
        self.safe_ui_update(_update)
    
    def update_image_display(self, pil_image, url, results, menu_name, properties_info, search_result_info=None):
        try:
            if not pil_image or self._cleanup_done:
                return
            
            display_size = (400, 300)
            img_copy = pil_image.copy()
            img_copy.thumbnail(display_size, Image.Resampling.LANCZOS)
            
            tk_image = ImageTk.PhotoImage(img_copy)
            
            if hasattr(self, 'image_label') and self.image_label.winfo_exists():
                self.image_label.config(image=tk_image, text="")
                self.image_label.image = tk_image
            
            if hasattr(self, 'url_text') and self.url_text.winfo_exists():
                self.url_text.config(state=tk.NORMAL)
                self.url_text.delete(1.0, tk.END)
                self.url_text.insert(tk.END, str(url))
                self.url_text.config(state=tk.DISABLED)
            
            if hasattr(self, 'result_text') and self.result_text.winfo_exists():
                self.result_text.config(state=tk.NORMAL)
                self.result_text.delete(1.0, tk.END)
                
                if results:
                    result_content = ""
                    for i, result in enumerate(results[:3]):
                        korean_name = self.translation_table.get(result['food'], result['food'])
                        result_content += f"{i+1}위: {korean_name} ({result['food']}) - 신뢰도: {result['confidence']:.3f}\n"
                    self.result_text.insert(tk.END, result_content)
                else:
                    self.result_text.insert(tk.END, "예측 결과 없음")
                
                self.result_text.config(state=tk.DISABLED)
            
            if hasattr(self, 'menu_info_text') and self.menu_info_text.winfo_exists():
                self.menu_info_text.config(state=tk.NORMAL)
                self.menu_info_text.delete(1.0, tk.END)
                
                menu_content = ""
                if menu_name:
                    menu_content += f"시트 메뉴명: {menu_name}\n"
                
                if results:
                    english_prediction = results[0]['food']
                    korean_mapped = self.translation_table.get(english_prediction, english_prediction)
                    menu_content += f"AI 예측: {korean_mapped}\n"
                    menu_content += f"신뢰도: {results[0]['confidence']:.3f}\n"
                    
                    if search_result_info:
                        menu_content += f"레시피 키워드: {search_result_info['used_keyword']} ({search_result_info['keyword_source']})\n"
                        menu_content += f"레시피 매칭: {search_result_info['found_menu']}\n"
                    else:
                        menu_content += f"레시피 매칭: 없음\n"
                
                self.menu_info_text.insert(tk.END, menu_content)
                self.menu_info_text.config(state=tk.DISABLED)
            
            if hasattr(self, 'recipe_text') and self.recipe_text.winfo_exists():
                self.recipe_text.config(state=tk.NORMAL)
                self.recipe_text.delete(1.0, tk.END)
                
                if search_result_info and properties_info:
                    found_menu = properties_info['menu_name']
                    properties = properties_info['properties']
                    used_keyword = search_result_info['used_keyword']
                    keyword_source = search_result_info['keyword_source']
                    
                    recipe_content = f"레시피 기준 메뉴: {found_menu}\n"
                    recipe_content += f"검색 키워드: {used_keyword} ({keyword_source})\n"
                    if found_menu != used_keyword:
                        recipe_content += f"매칭 결과: {used_keyword} → {found_menu}\n"
                    recipe_content += "\n"
                    
                    sheet_names = {
                        '주요식재료': '주요 식재료',
                        '양념소스': '양념 및 소스', 
                        '육수베이스': '육수/베이스',
                        '조리순서': '조리 순서',
                        '조리정보': '조리 정보',
                        '특징맛': '맛 특성',
                        '영양정보': '영양 정보',
                        '서빙조리팁': '조리 팁'
                    }
                    
                    for sheet_name, sheet_data in properties.items():
                        sheet_title = sheet_names.get(sheet_name, sheet_name)
                        recipe_content += f"\n--- {sheet_title} ---\n"
                        
                        count = 0
                        for ingredient, amount in sheet_data.items():
                            if count >= 15:
                                recipe_content += "  ... (더 많은 항목 있음)\n"
                                break
                            recipe_content += f"  • {ingredient}: {amount}\n"
                            count += 1
                        
                        recipe_content += "\n"
                else:
                    recipe_content = "레시피 정보를 찾을 수 없습니다.\n\n"
                    
                    if results:
                        english_prediction = results[0]['food']
                        korean_mapped = self.translation_table.get(english_prediction, english_prediction)
                        
                        recipe_content += f"AI 예측: {english_prediction} → {korean_mapped}\n"
                        
                        if korean_mapped == english_prediction:
                            recipe_content += f"매핑 상태: 매핑테이블에 '{english_prediction}' 없음\n"
                        else:
                            recipe_content += f"매핑 상태: 매핑테이블에서 '{korean_mapped}'로 변환됨\n"
                        
                        recipe_content += f"검색 결과: 속성파일에 '{korean_mapped}' 메뉴 없음\n\n"
                        recipe_content += "해결 방법:\n"
                        recipe_content += "1. 매핑테이블에 해당 영어명 추가\n"
                        recipe_content += "2. 속성파일에 해당 한글 메뉴명 추가"
                    else:
                        recipe_content += "예측 결과 정보가 없습니다."
                
                self.recipe_text.insert(tk.END, recipe_content)
                self.recipe_text.config(state=tk.DISABLED)
            
        except Exception as e:
            print(f"이미지 표시 업데이트 오류: {e}")
    
    def show_large_image(self, pil_image):
        try:
            img_window = tk.Toplevel(self.root)
            img_window.title("이미지 크게 보기")
            img_window.geometry("800x600")
            
            display_size = (750, 550)
            img_copy = pil_image.copy()
            img_copy.thumbnail(display_size, Image.Resampling.LANCZOS)
            
            tk_image = ImageTk.PhotoImage(img_copy)
            label = tk.Label(img_window, image=tk_image)
            label.image = tk_image
            label.pack(expand=True, fill=tk.BOTH)
            
            close_btn = tk.Button(img_window, text="닫기", command=img_window.destroy,
                                font=('Arial', 12), bg='#dc3545', fg='white')
            close_btn.pack(pady=10)
            
        except Exception as e:
            self.add_log(f"큰 이미지 표시 오류: {e}", "ERROR")
    
    def run(self):
        try:
            self.root.mainloop()
        except KeyboardInterrupt:
            print("사용자가 프로그램을 중단했습니다.")
        except Exception as e:
            print(f"GUI 실행 중 오류: {e}")
        finally:
            if not self._cleanup_done:
                self.on_closing()

def main():
    try:
        app = FoodPredictorGUI()
        app.run()
    except KeyboardInterrupt:
        print("사용자가 프로그램을 중단했습니다.")
    except Exception as e:
        print(f"프로그램 실행 중 오류: {e}")
    finally:
        try:
            cleanup_resources()
        except:
            pass

if __name__ == '__main__':
    main()